In [5]:
import os
from typing import Dict, Any
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.tools import tool
from langchain.agents import create_agent

import sys
sys.path.append(r"C:\My Projects\Health-Navigator")

from dotenv import load_dotenv
load_dotenv(r'C:\My Projects\Health-Navigator\credentials.env')

from app.workflow.vectordb.vectordb import HybridVectorDB

resource module not available on Windows


In [6]:
@tool
def retrieve_from_vector_db(
    user_id: str,
    query: str,
    top_k: int = 100,
    filters: dict = None,
    date: str = None,
    date_filter: str = None
) -> list:
    """
    Retrieves relevant medical records and health documents from a user's personal vector database.
    
    Uses hybrid search (semantic + BM25) to find the most relevant information based on the query.
    Each user has their own isolated database identified by user_id.
    
    Args:
        user_id (str): Unique identifier for the user whose database to search
        query (str): Search query to find relevant documents
        top_k (int, optional): Number of results to return. Defaults to 100, to make sure you return all relevant results.
        filters (dict, optional): Metadata filters to apply, e.g., {'type': 'prescription', 'doctor': 'Dr. Smith'}
        date (str, optional): Date for filtering in 'YYYY-MM-DD' format
        date_filter (str, optional): How to filter by date - 'before', 'at', or 'after'. Requires date parameter, you can only pass these three values ('before', 'at', 'after').
    
    Returns:
        list: List of dictionaries, each containing:
            - text (str): The retrieved document text
            - metadata (dict): Document metadata (type, date, doctor, etc.)
            - score (float): Relevance score
    
    Example:
        # Basic search
        results = retrieve_from_vector_db(user_id="user123", query="blood pressure medications")
        
        # Search with filters and date
        results = retrieve_from_vector_db(
            user_id="user123",
            query="lab results",
            top_k=5,
            filters={'type': 'lab_report'},
            date="2024-01-01",
            date_filter="after"
        )
    """
    db = HybridVectorDB(user_id=user_id)
    
    return db.retrieve(
        query=query,
        top_k=top_k,
        filters=filters,
        date=date,
        date_filter=date_filter
    )


@tool
def add_to_vector_db(
    user_id: str,
    text: str,
    metadata: dict = None
) -> bool:
    """
    Adds and indexes new medical text or health documents to a user's personal vector database.
    
    The function processes the text into nodes, extracts embeddings for semantic search, 
    and updates the BM25 index for hybrid retrieval. Dates in metadata are automatically 
    converted to integers for efficient filtering.
    
    Args:
        user_id (str): Unique identifier for the user whose database to update.
        text (str): The actual document content or medical note to be stored.
        metadata (dict, optional): Additional context such as {'type': 'prescription', 
                                   'date': '2024-05-20', 'doctor': 'Dr. Jordan'}.
    
    Returns:
        bool: True if the document was successfully indexed, False otherwise.
        
    Example:
        # Adding a new lab result
        success = add_to_vector_db(
            user_id="user123",
            text="Patient blood glucose levels are within normal range (95 mg/dL).",
            metadata={'type': 'lab_report', 'date': '2024-06-12'}
        )
    """
    try:
        db = HybridVectorDB(user_id=user_id)
        return db.add_text(text=text, metadata=metadata)
    except Exception as e:
        print(f"Failed to initialize database for user {user_id}: {e}")
        return False

@tool
def ask_user_for_info(request: str) -> str:
    """
    Request information from the user when it's not available in databases.
    Use this when you need subjective information, recent events not in records,
    current symptoms, or clarification that only the user can provide.
    
    Args:
        request: Specific question to ask the user (be clear and concise)
        
    Returns:
        User's response
    """
    # TODO: Implement actual user interaction via frontend
    print(f"\n[SYSTEM] User input needed: {request}")
    return "DUMMY_USER_RESPONSE"  # Replace with actual user input in production

# Initialize LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3-pro-preview",
    google_api_key=os.environ["GOOGLE_API_KEY"],
)

# Setup SQL Database
postgresql_uri = f'postgresql+psycopg2://{os.environ["POSTGRES_USERNAME"]}:{os.environ["POSTGRES_PASSWORD"]}@{os.environ["POSTGRES_HOST"]}:{os.environ["POSTGRES_PORT"]}/{os.environ["DATABASE_NAME"]}'
sql_db = SQLDatabase.from_uri(postgresql_uri)
sql_toolkit = SQLDatabaseToolkit(db=sql_db, llm=llm)
sql_tools = sql_toolkit.get_tools()

# DB Retriever Agent System Prompt
DB_RETRIEVER_SYSTEM_PROMPT = """You are a Medical Information Retrieval Specialist with access to vector databases, relational databases, and the ability to request information directly from users.

## Available Tools:
- **retrieve_from_vector_db**: Search medical knowledge base and patient documents
- **add_to_vector_db**: Store new information for future reference
- **SQL Database Tools**: Query structured patient records, lab results, medications, appointments, etc.
- **ask_user_for_info**: Request information directly from the user when not in databases

## Decision Framework:

### Query Database When:
- Patient's medical history, past diagnoses, procedures, medications
- Lab results, imaging reports, clinical notes
- Medical knowledge from guidelines or research
- Similar cases or treatment protocols
- Demographic or administrative data

### Use ask_user_for_info Tool When:
- Subjective information (current symptoms, pain levels, concerns)
- Recent events not yet documented
- Current home medications or lifestyle factors
- Personal preferences or context
- Clarification needed
- Family history not in records

## Workflow:
1. Start by querying databases for available information
2. Use ask_user_for_info tool if critical information is missing
3. Continue until you have comprehensive information
4. When complete, respond with:
```
INFORMATION_COMPLETE

**VECTOR DATABASE RESULTS:**
[Medical knowledge, research, guidelines]

**RELATIONAL DATABASE RESULTS:**
[Patient records, history, test results]

**USER PROVIDED INFORMATION:**
[Information gathered via ask_user_for_info tool]

**SUMMARY:** [Brief synthesis of all retrieved information]
```

## Guidelines:
- Always query databases BEFORE using ask_user_for_info
- Be specific in questions to users - explain why information is needed
- Current iteration: {reflection_count}/{max_reflections}
- At max iterations, work with available information
- Only retrieve and organize - do NOT provide medical advice

Remember: The Medical Agent handles clinical analysis. Your role is comprehensive information retrieval."""



# Combine SQL tools with vector DB tools
all_tools = sql_tools + [retrieve_from_vector_db, add_to_vector_db, ask_user_for_info]

# Create retriever agent


def invoke_db_retriever_agent(
    aggregated_output: str,
    info_request: str,
    reflection_count: int,
    max_reflections: int,
    user_id: str,
    conversation_history: list,
) -> Dict[str, Any]:
    """
    Invokes the DB retriever agent to gather information from databases.
    
    Returns:
        dict with keys: 'response', 'conversation_history', 'needs_more_info'
    """
    # Build initial query context
    if info_request:
        query_context = f"""
        Initial Medical Analysis Context:
        {aggregated_output}
        
        Medical Agent's Information Request:
        {info_request}
        
        Task: Retrieve the specific information requested by the Medical Agent.
        Determine if this information exists in databases or needs to be obtained from the user.
        """
    else:
        query_context = f"""
        Medical Analysis Context:
        {aggregated_output}
        
        Task: Gather comprehensive patient information from all available databases
        to support medical assessment.
        """
    
    # Format system prompt with current iteration info
    formatted_system_prompt = DB_RETRIEVER_SYSTEM_PROMPT.format(
        reflection_count=reflection_count,
        max_reflections=max_reflections
    )
    
    retriever_agent = create_agent(
    llm,
    tools=all_tools,
    system_prompt=formatted_system_prompt
)
    # Prepare messages for agent
    agent_input = {
        "messages": conversation_history + [
            HumanMessage(content=query_context)
        ]
    }
    
    # Invoke agent
    result = retriever_agent.invoke(agent_input)
    agent_response = result["messages"][-1].content
    
    # Determine if information is complete
    needs_more_info = "INFORMATION_COMPLETE" not in agent_response
    
    return {
        'response': agent_response,
        'conversation_history': result["messages"],
        'needs_more_info': needs_more_info
    }




In [7]:
# 1. DB Retriever Agent call
db_result = invoke_db_retriever_agent(
    aggregated_output="""
    Input Prompt: Patient experiencing chest pain and shortness of breath
    
    Numerical Analysis:
    Input: Heart rate: 105 bpm, Blood pressure: 145/90
    Output: Elevated vital signs detected
    
    Vision Analysis:
    Input: ECG image uploaded
    Output: Possible ST elevation observed
    """,
    info_request="",
    reflection_count=0,
    max_reflections=5,
    user_id="user_12345",
    conversation_history=[]
)


[SYSTEM] User input needed: Could you please provide the Patient ID or Name so I can retrieve their specific medical records?


In [8]:
print(db_result)

{'response': [{'type': 'text', 'text': 'INFORMATION_COMPLETE\n\n**VECTOR DATABASE RESULTS:**\n*   **Search Status:** Attempted searches with potential system/user identifiers (`1`, `0`, `system`, `guidelines`, `default`).\n*   **Findings:** No relevant medical knowledge or patient documents were retrieved. The database appears to contain no records for this context.\n\n**RELATIONAL DATABASE RESULTS:**\n*   **Patient Search:** The `users` table is empty (`count = 0`).\n*   **Records:** Consequently, `patient_profiles`, `medications`, `past_medical_history`, and other linked tables contain no data.\n*   **Conclusion:** No existing medical history, allergies, or past treatments are available for this patient.\n\n**USER PROVIDED INFORMATION:**\n*   **Chief Complaint:** Chest pain and shortness of breath.\n*   **Vitals:** Heart Rate 105 bpm (Tachycardia), Blood Pressure 145/90 mmHg (Hypertension).\n*   **Diagnostics:** ECG Image analysis indicates possible ST elevation.\n\n**SUMMARY:**\nThe

In [9]:
text = db_result['response'][0]['text']
history = db_result['conversation_history']
needs_more_info = db_result['needs_more_info']


In [10]:
print(text)

INFORMATION_COMPLETE

**VECTOR DATABASE RESULTS:**
*   **Search Status:** Attempted searches with potential system/user identifiers (`1`, `0`, `system`, `guidelines`, `default`).
*   **Findings:** No relevant medical knowledge or patient documents were retrieved. The database appears to contain no records for this context.

**RELATIONAL DATABASE RESULTS:**
*   **Patient Search:** The `users` table is empty (`count = 0`).
*   **Records:** Consequently, `patient_profiles`, `medications`, `past_medical_history`, and other linked tables contain no data.
*   **Conclusion:** No existing medical history, allergies, or past treatments are available for this patient.

**USER PROVIDED INFORMATION:**
*   **Chief Complaint:** Chest pain and shortness of breath.
*   **Vitals:** Heart Rate 105 bpm (Tachycardia), Blood Pressure 145/90 mmHg (Hypertension).
*   **Diagnostics:** ECG Image analysis indicates possible ST elevation.

**SUMMARY:**
The patient is presenting with acute chest pain and shortnes

In [11]:
print(history)

[HumanMessage(content='\n        Medical Analysis Context:\n        \n    Input Prompt: Patient experiencing chest pain and shortness of breath\n\n    Numerical Analysis:\n    Input: Heart rate: 105 bpm, Blood pressure: 145/90\n    Output: Elevated vital signs detected\n\n    Vision Analysis:\n    Input: ECG image uploaded\n    Output: Possible ST elevation observed\n    \n\n        Task: Gather comprehensive patient information from all available databases\n        to support medical assessment.\n        ', additional_kwargs={}, response_metadata={}, id='0a9367a1-3142-4868-80c6-25f690aacff2'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'sql_db_list_tables', 'arguments': '{"tool_input": ""}'}, '__gemini_function_call_thought_signatures__': {'7dd33ff7-852e-4a3e-ada0-5c7ad396c4fa': 'EuIQCt8QAXLI2nwxHp2/AbO+TxGBQnAxLUwxAjoSrEcoi++z46Wt7ZJLt5Moe6vg43gXdnl0oCbpEXRkby4ShcpBkenGhyjdTtwyPW+ZivTaSiZ3pCfg1y4BmBwPGyJfJ1cPl4Aas7P9yy0uxCfUDX3zw637gn5KROmHf1/wC2y4ZfGHWGdl67lV

In [12]:
print(needs_more_info)

True
